In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [52]:
loader = PyPDFLoader("../data/KedarkanthaKotgaonItinerary.pdf")
docs = loader.load()
len(docs)

2

In [53]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

5

In [54]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [55]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [56]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query=query)

In [ ]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
    
print(context)

In [58]:
llm = ChatOpenAI(model="gpt-5-nano")

#### Chai - Context generate | prompt | llm | strparser

In [59]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n" 
    
    return {
        "context": context,
        "question": query
    }
    

In [60]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answered based on the context for user question and
    if you don't know the answer, than you can say that 'I don't know.'
    Context: {context}
    Question: {question}                                   
""")

In [61]:
rag_chain = get_context | prompt | llm

In [66]:
res = rag_chain.invoke("Age limit for the trek?")

In [67]:
print(res.content)

Age limit: 8 to 62+ years. If you’re younger or older, please speak with us before registering.
